<a href="https://colab.research.google.com/github/MooreRA26/Language-Dataset-Mining-and-Merging/blob/main/JSONL_LanguageDatasetTraining.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Import and Install**

In [ ]:
!pip install click!=8.2.*,>=4.0
!pip install gtts
from gtts import gTTS

!pip install shapely
!pip install geopandas
!pip install cartopy
!pip install openai
get_ipython().system('pip install openai')
!pip install transformers==4.44.2
get_ipython().system('pip install transformers==4.44.2')
from transformers import pipeline
!pip install torch
get_ipython().system('pip install torch')
!pip install fastai
!pip install wikipedia
!pip install opencv-python

import tensorflow as tf
from tensorflow.keras.datasets import mnist




!pip install transformers==4.44.2

from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.callbacks import LearningRateScheduler
import numpy as np
import pandas as pd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import requests
import geopandas as gpd
from shapely.geometry import Point
from fastai.vision.all import *
from IPython.display import Audio
import ipywidgets as widgets
from IPython.display import display, Audio
from bs4 import BeautifulSoup
import cv2

import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

# Initialize tokenizer and model
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
model = GPT2LMHeadModel.from_pretrained("gpt2")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

!pip install transformers==4.44.2 torch


from transformers import AutoModelForCausalLM, AutoTokenizer

# Load model and tokenizer
fine_tuned_model = AutoModelForCausalLM.from_pretrained('gpt2')
tokenizer = AutoTokenizer.from_pretrained('gpt2')

# Move model to the correct device (CPU/GPU)
device = "cuda" if torch.cuda.is_available() else "cpu"
fine_tuned_model.to(device)

!pip install transformers==4.44.2 torch

# Set the pad_token to eos_token to avoid errors
tokenizer.pad_token = tokenizer.eos_token

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

from IPython.display import display, HTML
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.callbacks import LearningRateScheduler


from IPython.display import display, HTML

# **JSONL Merged Dataset Values (Important for finding out what values are present in order to tell the model how to train)**

In [ ]:
import json

path = "/content/merged_training_dataset.jsonl"

with open(path, "r", encoding="utf-8") as f:
    for i in range(5):
        line = f.readline().strip()
        obj = json.loads(line)
        print(f"\n--- line {i+1} ---")
        print("type:", type(obj))
        if isinstance(obj, dict):
            print("keys:", list(obj.keys()))
            # show a peek of values without dumping the whole thing
            for k in list(obj.keys())[:6]:
                v = obj[k]
                if isinstance(v, str):
                    print(k, "=", repr(v[:120]))
                else:
                    print(k, "type=", type(v))
        else:
            print(obj)


# **Training (Saves at checkpoints)**

In [ ]:
# akira_lm_train.py  (JSONL prompt/response only, disruption-hardened + ultra-robust checkpoint loading)
import os
import math
import json
import time
import random
import atexit
import signal
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Any

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import GPT2Tokenizer

# ============================================================
# CONFIG
# ============================================================
DATASET_PATH = "/content/merged_training_dataset.jsonl"
MODEL_SAVE_PATH = "/content/your_gpt_model_path_here.pth"   # required
BACKUP_SAVE_PATH = MODEL_SAVE_PATH + ".bak"
BAD_SAVE_PATH = MODEL_SAVE_PATH + ".bad"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SAVE_TO_DRIVE = False
DRIVE_SAVE_PATH = "/content/drive/MyDrive/your_gpt_model_path_here.pth"
DRIVE_BACKUP_PATH = DRIVE_SAVE_PATH + ".bak"
DRIVE_BAD_PATH = DRIVE_SAVE_PATH + ".bad"

MAX_LENGTH = 256
BATCH_SIZE = 16
NUM_EPOCHS = 5
LR = 2e-4
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
GRAD_ACCUM_STEPS = 1

USE_AMP = torch.cuda.is_available()
SAVE_EVERY_STEPS = 50
LOG_EVERY = 100

SEED = 42
IGNORE_INDEX = -100

D_MODEL = 256
N_HEAD = 8
N_LAYERS = 6
DIM_FF = 1024
DROPOUT = 0.1
MODEL_MAX_LEN = 2048

# Dataloader stability
# Colab CPU + tokenizers + multiprocessing can be flaky; 0 is most stable.
NUM_WORKERS = 0

# ============================================================
# TF32 control (CUDA-only)
# ============================================================
if torch.cuda.is_available():
    try:
        torch.backends.cuda.matmul.fp32_precision = "tf32"
        torch.backends.cudnn.fp32_precision = "tf32"
    except Exception:
        pass
torch.set_float32_matmul_precision("high")

# ============================================================
# AMP compatibility layer
# ============================================================
class NullScaler:
    def scale(self, loss): return loss
    def unscale_(self, optimizer): return None
    def step(self, optimizer): optimizer.step()
    def update(self): return None
    def state_dict(self): return {}
    def load_state_dict(self, state): return None

def get_grad_scaler():
    if not USE_AMP:
        return NullScaler()
    if hasattr(torch.cuda, "amp") and hasattr(torch.cuda.amp, "GradScaler"):
        return torch.cuda.amp.GradScaler()
    if hasattr(torch, "amp") and hasattr(torch.amp, "GradScaler"):
        try:
            return torch.amp.GradScaler()
        except TypeError:
            return NullScaler()
    return NullScaler()

def autocast_context():
    if not USE_AMP:
        from contextlib import nullcontext
        return nullcontext()
    if hasattr(torch.cuda, "amp") and hasattr(torch.cuda.amp, "autocast"):
        return torch.cuda.amp.autocast()
    if hasattr(torch, "amp") and hasattr(torch.amp, "autocast"):
        try:
            return torch.amp.autocast("cuda")
        except TypeError:
            return torch.amp.autocast()
    from contextlib import nullcontext
    return nullcontext()

# ============================================================
# Tokenizer
# ============================================================
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

PAD_ID = tokenizer.pad_token_id
EOS_ID = tokenizer.eos_token_id
VOCAB_SIZE = len(tokenizer)

print(f"PAD token: {tokenizer.pad_token} (id={PAD_ID})")
print(f"EOS token id: {EOS_ID}")
print(f"Vocab size: {VOCAB_SIZE}")
print(f"Using dataset: {DATASET_PATH}")
print(f"Device: {DEVICE}")

# ============================================================
# RNG helpers
# ============================================================
def seed_all(seed: int):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def get_rng_state() -> Dict[str, Any]:
    state = {"python_random_state": random.getstate(), "torch_rng_state": torch.get_rng_state()}
    if torch.cuda.is_available():
        state["cuda_rng_state_all"] = torch.cuda.get_rng_state_all()
    return state

def set_rng_state(state: Dict[str, Any]):
    if not state:
        return
    if "python_random_state" in state:
        random.setstate(state["python_random_state"])
    if "torch_rng_state" in state:
        torch.set_rng_state(state["torch_rng_state"])
    if torch.cuda.is_available() and "cuda_rng_state_all" in state:
        torch.cuda.set_rng_state_all(state["cuda_rng_state_all"])

seed_all(SEED)

# ============================================================
# Dataset
# ============================================================
class PromptResponseJSONLDataset(Dataset):
    def __init__(self, jsonl_path: str):
        if not os.path.exists(jsonl_path):
            raise FileNotFoundError(f"JSONL not found: {jsonl_path}")

        self.samples: List[Tuple[str, str]] = []
        bad_json = 0
        bad_schema = 0

        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    obj = json.loads(line)
                except json.JSONDecodeError:
                    bad_json += 1
                    continue

                if not isinstance(obj, dict):
                    bad_schema += 1
                    continue

                prompt = obj.get("prompt")
                response = obj.get("response")
                if not isinstance(prompt, str) or not isinstance(response, str):
                    bad_schema += 1
                    continue

                prompt = prompt.strip()
                response = response.strip()
                if not prompt or not response:
                    bad_schema += 1
                    continue

                prefix = f"User: {prompt}\nModel:"
                full = f"{prefix} {response}"
                self.samples.append((prefix, full))

        if not self.samples:
            raise ValueError("No usable samples loaded.")

        print(f"Loaded {len(self.samples)} prompt/response pairs from JSONL")
        if bad_json:
            print(f"Warning: skipped {bad_json} lines due to JSON parse errors")
        if bad_schema:
            print(f"Note: skipped {bad_schema} lines due to missing/invalid prompt/response")

    def __len__(self): return len(self.samples)
    def __getitem__(self, idx: int): return self.samples[idx]

# ============================================================
# Collate (FIXED): pad prefixes before making tensors
# ============================================================
@dataclass
class Batch:
    input_ids: torch.Tensor
    attention_mask: torch.Tensor
    labels: torch.Tensor

def collate_fn(examples: List[Tuple[str, str]]) -> Batch:
    prefixes, fulls = zip(*examples)

    # Full sequences (padded)
    enc_full = tokenizer(
        list(fulls),
        truncation=True,
        padding=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    input_ids = enc_full["input_ids"]
    attention_mask = enc_full["attention_mask"]

    # Prefix sequences (MUST be padded if return_tensors="pt")
    enc_pref = tokenizer(
        list(prefixes),
        truncation=True,
        padding=True,              # <-- FIX
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )
    # prefix length = number of real tokens in prefix
    prefix_lens = enc_pref["attention_mask"].sum(dim=1)

    # next-token labels
    labels = input_ids.clone()
    labels[:, :-1] = input_ids[:, 1:]
    labels[:, -1] = IGNORE_INDEX

    # ignore padding in loss
    labels = labels.masked_fill(attention_mask == 0, IGNORE_INDEX)

    # ignore prompt tokens: because labels are shifted, ignore labels up to prefix_len-1
    B = input_ids.size(0)
    for b in range(B):
        cut = int(prefix_lens[b].item())
        labels[b, :max(cut - 1, 0)] = IGNORE_INDEX

    return Batch(input_ids=input_ids, attention_mask=attention_mask, labels=labels)

# ============================================================
# Model
# ============================================================
class GPTBlock(nn.Module):
    def __init__(self, d_model: int, nhead: int, dim_ff: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, dim_ff), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(dim_ff, d_model), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor, attn_mask: torch.Tensor, key_padding_mask: torch.Tensor):
        h = self.ln1(x)
        a, _ = self.attn(h, h, h, attn_mask=attn_mask, key_padding_mask=key_padding_mask, need_weights=False)
        x = x + a
        x = x + self.ff(self.ln2(x))
        return x

class YourGPT(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, nhead: int, num_layers: int,
                 dim_ff: int, dropout: float, max_len: int, pad_id: int):
        super().__init__()
        self.pad_id = pad_id
        self.max_len = max_len
        self.tok_emb = nn.Embedding(vocab_size, d_model)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GPTBlock(d_model, nhead, dim_ff, dropout) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.tok_emb.weight

    def _causal_mask(self, S: int, device: torch.device) -> torch.Tensor:
        return torch.triu(torch.ones((S, S), dtype=torch.bool, device=device), diagonal=1)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
        B, S = input_ids.shape
        if S > self.max_len:
            raise ValueError(f"Sequence length {S} exceeds max_len {self.max_len}")

        pos = torch.arange(S, device=input_ids.device).unsqueeze(0).expand(B, S)
        x = self.tok_emb(input_ids) + self.pos_emb(pos)
        x = self.drop(x)

        causal = self._causal_mask(S, device=input_ids.device)
        key_padding_mask = (attention_mask == 0)

        for blk in self.blocks:
            x = blk(x, attn_mask=causal, key_padding_mask=key_padding_mask)

        x = self.ln_f(x)
        return self.lm_head(x)

# ============================================================
# Checkpointing
# ============================================================
_last_checkpoint_payload: Optional[Dict[str, Any]] = None

def atomic_torch_save(obj: Dict[str, Any], path: str, backup_path: Optional[str] = None):
    tmp = path + ".tmp"
    if backup_path is not None and os.path.exists(path):
        try:
            with open(path, "rb") as fsrc, open(backup_path + ".tmp", "wb") as fdst:
                fdst.write(fsrc.read())
            os.replace(backup_path + ".tmp", backup_path)
        except Exception:
            pass
    torch.save(obj, tmp)
    os.replace(tmp, path)

def maybe_copy_to_drive(src: str, dst: str, dst_bak: Optional[str] = None):
    if not SAVE_TO_DRIVE:
        return
    try:
        if dst_bak is not None and os.path.exists(dst):
            with open(dst, "rb") as fsrc, open(dst_bak + ".tmp", "wb") as fdst:
                fdst.write(fsrc.read())
            os.replace(dst_bak + ".tmp", dst_bak)
        with open(src, "rb") as fsrc, open(dst + ".tmp", "wb") as fdst:
            fdst.write(fsrc.read())
        os.replace(dst + ".tmp", dst)
        print(f"Copied checkpoint → {dst}")
    except Exception as e:
        print(f"[warn] Could not copy to Drive: {e}")

def save_checkpoint(path: str, model: nn.Module, optimizer: torch.optim.Optimizer, scaler,
                    epoch: int, step_in_epoch: int, global_step: int, permutation: torch.Tensor):
    payload = {
        "format_version": 5,
        "meta": {"timestamp": time.time(), "dataset_path": DATASET_PATH, "max_length": MAX_LENGTH, "seed": SEED},
        "tokenizer": {"name": "gpt2", "pad_token": tokenizer.pad_token, "pad_id": PAD_ID, "eos_id": EOS_ID, "vocab_size": VOCAB_SIZE},
        "model_config": {"vocab_size": VOCAB_SIZE, "d_model": D_MODEL, "nhead": N_HEAD, "num_layers": N_LAYERS,
                         "dim_ff": DIM_FF, "dropout": DROPOUT, "max_len": MODEL_MAX_LEN, "pad_id": PAD_ID},
        "train_state": {"epoch": epoch, "step_in_epoch": step_in_epoch, "global_step": global_step, "grad_accum_steps": GRAD_ACCUM_STEPS},
        "permutation": permutation.cpu(),
        "rng_state": get_rng_state(),
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scaler_state_dict": scaler.state_dict() if (USE_AMP and hasattr(scaler, "state_dict")) else None,
    }
    global _last_checkpoint_payload
    _last_checkpoint_payload = payload
    atomic_torch_save(payload, path, backup_path=BACKUP_SAVE_PATH)
    print(f"Checkpoint saved → {path}")
    maybe_copy_to_drive(path, DRIVE_SAVE_PATH, dst_bak=DRIVE_BACKUP_PATH)

def _is_dummy_checkpoint(d: Dict[str, Any]) -> bool:
    keys = list(d.keys())
    return set(keys) <= {"dummy.weight", "dummy.bias"} and len(keys) <= 2

def quarantine_bad_checkpoint(local_path: str):
    try:
        if os.path.exists(local_path):
            os.replace(local_path, BAD_SAVE_PATH)
            print(f"[warn] Moved unusable checkpoint to: {BAD_SAVE_PATH}")
    except Exception as e:
        print(f"[warn] Could not move bad checkpoint: {e}")

def load_checkpoint_if_exists(path: str, model: nn.Module, optimizer=None, scaler=None):
    if not os.path.exists(path):
        return None

    print(f"Loading checkpoint from {path}")
    ckpt = torch.load(path, map_location=DEVICE)

    # payload resumable
    if isinstance(ckpt, dict) and "model_state_dict" in ckpt and "train_state" in ckpt:
        model.load_state_dict(ckpt["model_state_dict"], strict=True)
        if optimizer is not None and ckpt.get("optimizer_state_dict") is not None:
            optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        if scaler is not None and ckpt.get("scaler_state_dict") is not None:
            try: scaler.load_state_dict(ckpt["scaler_state_dict"])
            except Exception: pass
        set_rng_state(ckpt.get("rng_state", {}))
        return ckpt

    # dummy OrderedDict checkpoint case
    if isinstance(ckpt, dict) and _is_dummy_checkpoint(ckpt):
        print("[warn] Checkpoint is a dummy OrderedDict (dummy.weight/dummy.bias). Ignoring and starting fresh.")
        quarantine_bad_checkpoint(path)
        return None

    # weights-only state_dict case
    if isinstance(ckpt, dict) and all(isinstance(k, str) for k in ckpt.keys()) and all(torch.is_tensor(v) for v in ckpt.values()):
        missing, unexpected = model.load_state_dict(ckpt, strict=False)
        if missing:
            print(f"[warn] Missing keys from legacy checkpoint: {len(missing)} (up to 8): {missing[:8]}")
        if unexpected:
            print(f"[warn] Unexpected keys from legacy checkpoint: {len(unexpected)} (up to 8): {unexpected[:8]}")
        print("[info] Loaded LEGACY weights-only checkpoint. Optimizer/epoch not resumable from this file.")
        return {"train_state": {"epoch": 1, "step_in_epoch": 0, "global_step": 0}, "permutation": None, "legacy_loaded": True}

    print("\n[debug] Unrecognized checkpoint format.")
    print(f"[debug] type(ckpt) = {type(ckpt)}")
    if isinstance(ckpt, dict):
        keys = list(ckpt.keys())
        print(f"[debug] dict keys (up to 60): {keys[:60]}")
        for k in keys[:10]:
            try: print(f"[debug] key '{k}' -> type {type(ckpt[k])}")
            except Exception: pass
    raise ValueError("Checkpoint exists but format is unrecognized (see [debug] output above).")

# atexit + signals
def _atexit_save():
    global _last_checkpoint_payload
    if _last_checkpoint_payload is None:
        return
    try:
        atomic_torch_save(_last_checkpoint_payload, MODEL_SAVE_PATH, backup_path=BACKUP_SAVE_PATH)
        print(f"[atexit] Wrote last checkpoint → {MODEL_SAVE_PATH}")
        maybe_copy_to_drive(MODEL_SAVE_PATH, DRIVE_SAVE_PATH, dst_bak=DRIVE_BACKUP_PATH)
    except Exception as e:
        print(f"[atexit warn] Could not save checkpoint: {e}")

atexit.register(_atexit_save)

def _signal_handler(signum, frame):
    print(f"\n[signal] Caught signal {signum}. Attempting emergency checkpoint...")
    _atexit_save()
    raise KeyboardInterrupt

signal.signal(signal.SIGINT, _signal_handler)
signal.signal(signal.SIGTERM, _signal_handler)

# ============================================================
# Dataloader with exact resume support
# ============================================================
class PermutedSubset(Dataset):
    def __init__(self, base_ds: Dataset, perm: torch.Tensor):
        self.base_ds = base_ds
        self.perm = perm
    def __len__(self): return self.perm.numel()
    def __getitem__(self, i: int): return self.base_ds[int(self.perm[i].item())]

def make_epoch_permutation(n: int, epoch: int) -> torch.Tensor:
    g = torch.Generator()
    g.manual_seed(SEED + epoch * 1000 + 7)
    return torch.randperm(n, generator=g)

def build_loader_for_epoch(base_ds: Dataset, perm: torch.Tensor, start_step_in_epoch: int) -> Tuple[DataLoader, int]:
    start_item = start_step_in_epoch * BATCH_SIZE
    if start_item >= len(perm):
        start_item = len(perm)
    subset = PermutedSubset(base_ds, perm[start_item:])
    loader = DataLoader(
        subset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        drop_last=False,
        num_workers=NUM_WORKERS,  # stable
        pin_memory=torch.cuda.is_available(),
        collate_fn=collate_fn,
    )
    return loader, start_item

# ============================================================
# Generation
# ============================================================
@torch.no_grad()
def generate(model: nn.Module, prompt: str, max_new_tokens: int = 80, temperature: float = 0.9, top_k: int = 50, top_p: float = 0.95) -> str:
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt")
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = torch.ones_like(input_ids, device=DEVICE)

    for _ in range(max_new_tokens):
        logits = model(input_ids, attention_mask)[:, -1, :]
        logits = logits / max(temperature, 1e-6)

        if top_k and top_k > 0:
            k = min(top_k, logits.size(-1))
            values, _ = torch.topk(logits, k)
            cutoff = values[:, -1].unsqueeze(1)
            logits = torch.where(logits < cutoff, torch.full_like(logits, float("-inf")), logits)

        if top_p and 0 < top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            probs = torch.softmax(sorted_logits, dim=-1)
            cumprobs = probs.cumsum(dim=-1)
            cutoff = cumprobs > top_p
            cutoff[:, 1:] = cutoff[:, :-1].clone()
            cutoff[:, 0] = False
            sorted_logits = torch.where(cutoff, torch.full_like(sorted_logits, float("-inf")), sorted_logits)
            logits = torch.full_like(logits, float("-inf")).scatter(1, sorted_indices, sorted_logits)

        probs = torch.softmax(logits, dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)

        input_ids = torch.cat([input_ids, next_id], dim=1)
        attention_mask = torch.ones_like(input_ids, device=DEVICE)

        if next_id.item() == EOS_ID:
            break
        if input_ids.size(1) >= MODEL_MAX_LEN:
            break

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)

# ============================================================
# TRAINING
# ============================================================
def train():
    if not os.path.exists(DATASET_PATH):
        raise FileNotFoundError(f"Dataset not found at: {DATASET_PATH}")

    base_ds = PromptResponseJSONLDataset(DATASET_PATH)

    model = AkiraGPT(VOCAB_SIZE, D_MODEL, N_HEAD, N_LAYERS, DIM_FF, DROPOUT, MODEL_MAX_LEN, PAD_ID).to(DEVICE)
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scaler = get_grad_scaler()

    start_epoch = 1
    start_step_in_epoch = 0
    global_step = 0
    perm_for_epoch = None

    ckpt = load_checkpoint_if_exists(MODEL_SAVE_PATH, model, optimizer, scaler)
    if ckpt is not None:
        ts = ckpt.get("train_state", {})
        start_epoch = int(ts.get("epoch", 1))
        start_step_in_epoch = int(ts.get("step_in_epoch", 0))
        global_step = int(ts.get("global_step", 0))
        perm_for_epoch = ckpt.get("permutation", None)
        print(f"Resuming from epoch {start_epoch}, step_in_epoch {start_step_in_epoch}, global_step {global_step}")

    model.train()

    try:
        epoch = start_epoch
        while epoch <= NUM_EPOCHS:
            if not isinstance(perm_for_epoch, torch.Tensor) or perm_for_epoch.numel() != len(base_ds):
                perm_for_epoch = make_epoch_permutation(len(base_ds), epoch)

            loader, start_item = build_loader_for_epoch(base_ds, perm_for_epoch, start_step_in_epoch)

            print(f"\n=== Epoch {epoch}/{NUM_EPOCHS} ===")
            if start_step_in_epoch > 0:
                print(f"Continuing epoch at batch {start_step_in_epoch} (item offset {start_item})")

            token_loss_sum = 0.0
            token_count = 0

            optimizer.zero_grad(set_to_none=True)
            step_in_epoch = start_step_in_epoch

            for batch_idx, batch in enumerate(loader, 1):
                input_ids = batch.input_ids.to(DEVICE, non_blocking=True)
                attention_mask = batch.attention_mask.to(DEVICE, non_blocking=True)
                labels = batch.labels.to(DEVICE, non_blocking=True)

                with autocast_context():
                    logits = model(input_ids, attention_mask)
                    B, S, V = logits.shape
                    loss = F.cross_entropy(logits.view(B * S, V), labels.view(B * S), ignore_index=IGNORE_INDEX)
                    loss = loss / GRAD_ACCUM_STEPS

                scaler.scale(loss).backward()

                with torch.no_grad():
                    valid_tokens = (labels != IGNORE_INDEX).sum().item()
                    token_loss_sum += (loss.item() * GRAD_ACCUM_STEPS) * valid_tokens
                    token_count += valid_tokens

                if (batch_idx % GRAD_ACCUM_STEPS) == 0:
                    scaler.unscale_(optimizer)
                    if GRAD_CLIP is not None:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                    scaler.step(optimizer)
                    scaler.update()
                    optimizer.zero_grad(set_to_none=True)

                global_step += 1
                step_in_epoch += 1

                if LOG_EVERY and global_step % LOG_EVERY == 0:
                    avg = token_loss_sum / max(token_count, 1)
                    print(f"step {global_step} | epoch {epoch} | step_in_epoch {step_in_epoch} | avg_token_loss {avg:.4f}")

                if SAVE_EVERY_STEPS and (global_step % SAVE_EVERY_STEPS == 0):
                    save_checkpoint(MODEL_SAVE_PATH, model, optimizer, scaler, epoch, step_in_epoch, global_step, perm_for_epoch)

            epoch_avg = token_loss_sum / max(token_count, 1)
            ppl = math.exp(epoch_avg) if epoch_avg < 20 else float("inf")
            print(f"Epoch {epoch} done | token-avg loss: {epoch_avg:.4f} | ppl: {ppl:.2f}")

            save_checkpoint(MODEL_SAVE_PATH, model, optimizer, scaler, epoch + 1, 0, global_step, torch.empty(0))

            epoch += 1
            start_step_in_epoch = 0
            perm_for_epoch = None

    except KeyboardInterrupt:
        print("\nTraining interrupted. Saving checkpoint...")
        try:
            if not isinstance(perm_for_epoch, torch.Tensor) or perm_for_epoch.numel() != len(base_ds):
                perm_for_epoch = make_epoch_permutation(len(base_ds), epoch)
            save_checkpoint(MODEL_SAVE_PATH, model, optimizer, scaler, epoch, step_in_epoch, global_step, perm_for_epoch)
        except Exception as e:
            print(f"[warn] Could not save on interrupt: {e}")
        raise

    sample_prompt = "User: I'm feeling overwhelmed.\nModel:"
    out = generate(model, sample_prompt)
    print("\n=== SAMPLE ===")
    print(out)

# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    train()
